# Model Training Notebook

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, roc_auc_score

# --- Import your custom modules ---
import util
import indicatorBuilder as ind

## 1. Get and Prepare Data

In [7]:
cl_data = util.get_cl_data('data/test100k.csv')
print("--- Original Data Head ---")
print(cl_data.head())

Successfully read data/test100k.csv using separator: ','
--- Original Data Head ---
                           Open        High         Low       Close  Volume
DateTime                                                                   
2008-11-21 00:00:00  184.754135  185.162883  184.679817  185.014247      75
2008-11-21 00:05:00  185.125724  185.274360  185.051406  185.088565      38
2008-11-21 00:10:00  185.162883  185.237201  185.051406  185.125724      27
2008-11-21 00:15:00  185.051406  185.868903  185.014247  185.868903     234
2008-11-21 00:20:00  185.868903  186.129015  185.645949  185.720267      74


## 2. Generate Indicator Features

In [10]:
features_with_indicators = ind.generate_features(cl_data)
print("\n--- Data with Features Head ---")
# Increase the number of rows printed to ensure you see non-NaN indicator values
print(features_with_indicators.head(30)) 


--- Data with Features Head ---
                           Open        High         Low       Close  Volume  \
DateTime                                                                      
2009-01-07 03:05:00  180.139577  180.325749  179.916171  180.251280     117   
2009-01-07 03:10:00  180.176811  180.176811  179.283188  179.320422     250   
2009-01-07 03:15:00  179.283188  179.841703  179.097017  179.730000     171   
2009-01-07 03:20:00  179.692765  179.804468  179.283188  179.581063      56   
2009-01-07 03:25:00  179.581063  179.618297  179.357657  179.618297      94   
2009-01-07 03:30:00  179.543828  179.841703  179.208720  179.357657     106   
2009-01-07 03:35:00  179.208720  179.208720  177.979987  178.203393     262   
2009-01-07 03:40:00  178.166159  178.724674  177.979987  178.538502     235   
2009-01-07 03:45:00  178.575736  178.985314  178.426799  178.464033     122   
2009-01-07 03:50:00  178.575736  178.724674  178.315096  178.426799      65   
2009-01-07 03:55:00

## 3. Define Target Variable (y) and Features (X)

In [13]:
# --- OPTION 1: REGRESSION (Predict next closing price) ---
# To use this, uncomment the two lines below and comment out the Classification block.
# TARGET_MODE = 'regression'
# features_with_indicators['target'] = features_with_indicators['Close'].shift(-1)

# --- OPTION 2: CLASSIFICATION (Predict >5% price change in next 3 days) ---
# This is the currently active mode.
TARGET_MODE = 'classification'
# Calculate the number of 5-minute periods in 3 days
periods_in_3_days = 12 * 24 * 3  # (12 5-min periods per hour) * (24 hours) * (3 days)
# Look ahead to get the price 3 days from now
future_price = features_with_indicators['Close'].shift(-periods_in_3_days)
# Calculate the percentage change
price_change_pct = (future_price - features_with_indicators['Close']) / features_with_indicators['Close']
# Set the target: 1 if the absolute change is > 5%, otherwise 0
features_with_indicators['target'] = (np.abs(price_change_pct) > 0.05).astype(int)

# --- Data Cleaning and Finalizing X and y ---
# Drop rows with any NaN values. This is crucial as NaNs are created by:
# 1. Initial indicator calculations (e.g., a 20-period SMA is NaN for the first 19 rows).
# 2. The target variable lookahead (the last 'periods_in_3_days' rows will be NaN).
final_df = features_with_indicators.dropna()

# Define features (X) and target (y)
X = final_df.drop(columns=['Open', 'High', 'Low', 'Close', 'Volume', 'target'])
y = final_df['target']

print("\n--- Final Features (X) Head ---")
print(X.head())
print("\n--- Final Target (y) Head ---")
print(y.head())
print(f"\nTarget Mode: {TARGET_MODE}")

if TARGET_MODE == 'classification':
    print("Target Distribution (0s and 1s):")
    # Check for class imbalance. If one class is rare, you may need techniques like SMOTE.
    print(y.value_counts(normalize=True))


--- Final Features (X) Head ---
                           RSI      SMA_20      EMA_20   VOL_24H    VOL_5D  \
DateTime                                                                     
2009-01-07 03:05:00  53.892113  179.722553  179.948818  0.003318  0.003625   
2009-01-07 03:10:00  45.231916  179.798883  179.888971  0.003331  0.003627   
2009-01-07 03:15:00  49.107144  179.867767  179.873831  0.003331  0.003628   
2009-01-07 03:20:00  47.783123  179.932927  179.845948  0.003322  0.003626   
2009-01-07 03:25:00  48.159429  179.953406  179.824267  0.003322  0.003624   

                      VOL_30D  
DateTime                       
2009-01-07 03:05:00  0.003614  
2009-01-07 03:10:00  0.003615  
2009-01-07 03:15:00  0.003615  
2009-01-07 03:20:00  0.003615  
2009-01-07 03:25:00  0.003615  

--- Final Target (y) Head ---
DateTime
2009-01-07 03:05:00    1
2009-01-07 03:10:00    1
2009-01-07 03:15:00    1
2009-01-07 03:20:00    1
2009-01-07 03:25:00    1
Name: target, dtype: int32

Tar

## 4. Split Data into Training and Testing Sets

In [16]:
# For time series, we perform a chronological split to avoid data leakage.
test_size = 0.2
split_index = int(len(X) * (1 - test_size))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"\nTraining set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")


Training set size: 73088 samples
Testing set size: 18273 samples


## 5. Create and Train the LightGBM Model

In [19]:
if TARGET_MODE == 'regression':
    print("\n--- Training LightGBM REGRESSOR ---")
    model = lgb.LGBMRegressor(objective='regression', metric='rmse', n_estimators=1000, seed=42)
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(100, verbose=True)])
    
elif TARGET_MODE == 'classification':
    print("\n--- Training LightGBM CLASSIFIER ---")
    # is_unbalance=True can be helpful if the target classes are imbalanced.
    model = lgb.LGBMClassifier(objective='binary', metric='logloss', n_estimators=1000, seed=42, is_unbalance=True)
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              eval_metric='logloss',
              callbacks=[lgb.early_stopping(100, verbose=True)])


--- Training LightGBM CLASSIFIER ---
[LightGBM] [Info] Number of positive: 22154, number of negative: 50934
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000863 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 73088, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303114 -> initscore=-0.832513
[LightGBM] [Info] Start training from score -0.832513
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[22]	valid_0's binary_logloss: 0.252266


## 6. Make Predictions and Evaluate

In [24]:
if TARGET_MODE == 'regression':
    print("\n--- Making Predictions on Test Data ---")
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(f"\nModel Evaluation (RMSE): {rmse:.4f}")
    
    results = pd.DataFrame({'Actual': y_test, 'Predicted': predictions})
    print("\n--- Sample of Predictions vs Actual Values ---")
    print(results.head(10))

elif TARGET_MODE == 'classification':
    print("\n--- Making Predictions on Test Data ---")
    predictions = model.predict(X_test)
    pred_probs = model.predict_proba(X_test)[:, 1]  # type: ignore # Probabilities for the '1' class
    
    accuracy = accuracy_score(y_test, predictions)
    auc = roc_auc_score(y_test, pred_probs)
    
    print(f"\nModel Evaluation:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC (Area Under Curve): {auc:.4f}")
    
    results = pd.DataFrame({'Actual': y_test, 'Predicted_Class': predictions, 'Predicted_Prob_of_1': pred_probs})
    print("\n--- Sample of Predictions vs Actual Values ---")
    print(results.head(10))


--- Making Predictions on Test Data ---

Model Evaluation:
Accuracy: 0.8991
AUC (Area Under Curve): 0.7602

--- Sample of Predictions vs Actual Values ---
                     Actual  Predicted_Class  Predicted_Prob_of_1
DateTime                                                         
2010-01-21 05:50:00       0                0             0.121506
2010-01-21 05:55:00       0                0             0.121506
2010-01-21 06:00:00       0                0             0.121506
2010-01-21 06:05:00       0                0             0.121506
2010-01-21 06:10:00       0                0             0.121506
2010-01-21 06:15:00       0                0             0.121506
2010-01-21 06:20:00       0                0             0.121506
2010-01-21 06:25:00       0                0             0.121506
2010-01-21 06:30:00       0                0             0.121506
2010-01-21 06:35:00       0                0             0.121506
